# 08 - Duplicatas

## Objetivo
Identificar, remover e analisar linhas duplicadas.

## Conceitos

### O que sao duplicatas
Linhas identicas em todas as colunas, ou em um subconjunto de colunas.
Podem surgir de erros de entrada, juncoes mal feitas ou integracoes.

### Deteccao
- `df.duplicated()`: Series booleana marcando duplicatas (exceto a primeira).
- `df.duplicated(keep=False)`: marca todas as ocorrencias.
- `df.duplicated(subset=["col"])`: considera apenas colunas indicadas.
- `df.duplicated().sum()`: total de duplicatas.

### Remocao
- `df.drop_duplicates()`: remove linhas duplicadas.
- `df.drop_duplicates(subset=["col"])`: por subconjunto.
- `df.drop_duplicates(keep="last")`: mantem a ultima ocorrencia.
- `df.drop_duplicates(keep=False)`: remove todas as duplicatas.

### Cuidados
Antes de remover, entenda por que existem. Duplicatas podem ser
legitimas (mesma venda registrada duas vezes) ou erros.
Sempre investigue com `duplicated(keep=False)`.

### Apos merge
Merges podem multiplicar linhas. Use `validate` para prevenir.

## DataFrame de exemplo com duplicatas
Criamos um DataFrame onde os registros de `Bruno` e `Diego` aparecem
duas vezes (linhas identicas em todas as colunas).

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "id": [1, 2, 2, 3, 4, 4, 5],
    "nome": ["Ana", "Bruno", "Bruno", "Carla", "Diego", "Diego", "Elisa"],
    "cidade": ["SP", "RJ", "RJ", "MG", "SP", "SP", "RJ"],
})
print("DataFrame:\n", df)

## Deteccao de duplicatas
- `duplicated()`: marca como `True` as linhas que repetem uma anterior
  (a **primeira** ocorrencia fica `False`).
- `duplicated(keep=False)`: marca **todas** as ocorrencias como `True`,
  permitindo filtrar os grupos duplicados completos.

In [ ]:
# Deteccao
print("\nduplicated:\n", df.duplicated())
print("Total de duplicatas:", df.duplicated().sum())

# Marcando todas as ocorrencias
print("\nduplicated(keep=False):\n", df.duplicated(keep=False))

# Filtrando duplicatas
print("\nLinhas duplicadas:\n", df[df.duplicated(keep=False)])

## Remocao com drop_duplicates
- `drop_duplicates()`: mantem a **primeira** ocorrencia e remove as demais.
- `keep="last"`: mantem a **ultima** ocorrencia.
- `keep=False`: remove **todas** as linhas que participam de algum grupo duplicado.

In [ ]:
# Remocao basica
print("\ndrop_duplicates():\n", df.drop_duplicates())

# Mantendo a ultima
print("\nkeep='last':\n", df.drop_duplicates(keep="last"))

# Removendo todas as duplicatas
print("\nkeep=False:\n", df.drop_duplicates(keep=False))

## Duplicatas em subconjunto de colunas
`subset=["col"]` considera apenas as colunas indicadas para decidir
o que e duplicata. Util quando a chave natural esta em algumas colunas,
mas nao em todas.

In [ ]:
# Subconjunto de colunas
print("\nSubset id:\n", df.drop_duplicates(subset=["id"]))

# Verificando unicidade
print("\nid unico?", df["id"].is_unique)

# Contagem de ocorrencias
print("\nvalue_counts em nome:\n", df["nome"].value_counts())

## Mantendo a linha com maior valor
Uma estrategia comum: ordenar pelo valor de interesse (decrescente) e
depois remover duplicatas pela chave, mantendo apenas o topo de cada grupo.

In [ ]:
# Mantendo a linha com maior valor em uma coluna
df2 = pd.DataFrame({
    "id": [1, 1, 2, 2],
    "valor": [10, 20, 5, 15],
})
print("\ndf2:\n", df2)
print("Maior valor por id:\n",
      df2.sort_values("valor", ascending=False)
         .drop_duplicates(subset=["id"]))

## Normalizacao antes de detectar duplicatas
Emails com diferenca de maiusculas/espacos podem passar como distintos.
Normalizar (lower + strip) antes de checar duplicatas evita falsos negativos.

In [ ]:
# Duplicatas em strings
df3 = pd.DataFrame({
    "email": ["a@x.com", "A@X.COM", "b@x.com", "a@x.com"],
})
print("\ndf3:\n", df3)
df3["email_norm"] = df3["email"].str.lower().str.strip()
print("Apos normalizar:\n", df3)
print("Duplicatas apos normalizar:",
      df3.duplicated(subset=["email_norm"]).sum())

## Reset apos remocao
Apos `drop_duplicates`, o indice mantem os rotulos originais (com buracos).
`reset_index(drop=True)` recria um indice sequencial.

In [ ]:
# Reset apos remocao
limpo = df.drop_duplicates().reset_index(drop=True)
print("\nLimpo com indice resetado:\n", limpo)